Load libraries

In [ ]:
import torch
import torch.nn as nn
from network_generator.config import NetworkConfig, TargetConfig, TrainingConfig
from network_generator.data_iot import (
    load_input_output_table,
    assign_firms_to_industries,
)
from network_generator.diagnostics import compute_network_stats
from network_generator.initialization import (
    directed_configuration_model,
    generate_burr_degrees,
    generate_burr_degrees_matching_sum,
    generate_initial_strength_fitnesses,
    get_parameters_for_burr_distribution,
    populate_weights_fourparams,
    triangular_wedge_matching,
)
from network_generator.metrics import (
    compute_degrees_from_model,
    compute_hill_exponent,
)
from network_generator.utils import scale_by_iot
from network_generator.visualization import visualize_network

criterion_sse = nn.MSELoss(reduction="sum")
criterion_mse = nn.MSELoss()

Load configurations

In [ ]:
# Configuration - modify these config classes to change parameters
network_config = NetworkConfig(num_firms=100_000)
target_config = TargetConfig()
training_config = TrainingConfig()

# For backward compatibility, extract values to individual variables
num_firms = network_config.num_firms
monetary_unit = network_config.monetary_unit
country_code = network_config.country_code
country_year = network_config.country_year

Load target values from config into tensors

In [ ]:
# Extract target values from config
target_hill_in_deg = torch.tensor(target_config.target_hill_in_deg, dtype=torch.float32)
target_hill_out_deg = torch.tensor(target_config.target_hill_out_deg, dtype=torch.float32)
target_hill_weight_distribution = torch.tensor(target_config.target_hill_weight_distribution, dtype=torch.float32)
target_hill_in_str = torch.tensor(target_config.target_hill_in_str, dtype=torch.float32)
target_hill_out_str = torch.tensor(target_config.target_hill_out_str, dtype=torch.float32)
target_average_connection = torch.tensor(target_config.target_average_connection, dtype=torch.float32)

target_so_ko_corr = torch.tensor(target_config.target_so_ko_corr, dtype=torch.float32)
target_si_ki_corr = torch.tensor(target_config.target_si_ki_corr, dtype=torch.float32)
target_si_so_corr = torch.tensor(target_config.target_si_so_corr, dtype=torch.float32)
target_ki_ko_corr = torch.tensor(target_config.target_ki_ko_corr, dtype=torch.float32)

target_so_ko_regcoeff = torch.tensor(target_config.target_so_ko_regcoeff, dtype=torch.float32)
target_ko_so_regcoeff = torch.tensor(target_config.target_ko_so_regcoeff, dtype=torch.float32)
target_si_ki_regcoeff = torch.tensor(target_config.target_si_ki_regcoeff, dtype=torch.float32)
target_ki_si_regcoeff = torch.tensor(target_config.target_ki_si_regcoeff, dtype=torch.float32)

target_tls_str = torch.tensor(target_config.target_tls_str, dtype=torch.float32)
target_tls_deg = torch.tensor(target_config.target_tls_deg, dtype=torch.float32)

Load parameters for Burr in- and out-degree sequences

In [ ]:
# Extract Burr distribution parameters from network_config
burr_max_indeg = torch.tensor(network_config.burr_max_indeg, dtype=torch.float32)
burr_max_outdeg = torch.tensor(network_config.burr_max_outdeg, dtype=torch.float32)

burr_alpha_outdeg = network_config.burr_alpha_outdeg
burr_var_log_outdeg = network_config.burr_var_log_outdeg

burr_alpha_indeg = network_config.burr_alpha_indeg
burr_var_log_indeg = network_config.burr_var_log_indeg

k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(burr_alpha_outdeg, burr_var_log_outdeg, target_average_connection)
k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(burr_alpha_indeg, burr_var_log_indeg, target_average_connection)

Generate Burr out-degree sequence

In [ ]:
out_deg_seq_burr = generate_burr_degrees(
    c_val=c_outdeg,
    d_val=k_outdeg,
    lam_val=lam_outdeg,
    max_cutoff=burr_max_outdeg,
    num_of_firms=num_firms,
)
itr = 0
while (
    (out_deg_seq_burr.max().item() < 0.1 * num_firms)
    or (out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item() < 2.95)
    or (
        compute_hill_exponent(
            out_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
        > 1.5
    )
    or (
        compute_hill_exponent(
            out_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
        < 1.4
    )
    or (out_deg_seq_burr.float().mean() < 40.0)
):
    out_deg_seq_burr = generate_burr_degrees(
        c_val=c_outdeg,
        d_val=k_outdeg,
        lam_val=lam_outdeg,
        max_cutoff=burr_max_outdeg,
        num_of_firms=num_firms,
    )
    itr += 1
    print(itr)
out_deg_seq_burr.max(), out_deg_seq_burr.float().mean(), out_deg_seq_burr[
    out_deg_seq_burr > 0
].log().var(), compute_hill_exponent(
    out_deg_seq_burr, network_config.threshold_percentile_for_hill_deg, is_degree=True
)

Generate Burr in-degree sequence, whose sum matches the sum of the generated Burr out-degree seqeuence

In [ ]:
itr = 0
in_deg_seq_burr = generate_burr_degrees_matching_sum(
    c_val=c_indeg,
    d_val=k_indeg,
    lam_val=lam_indeg,
    max_cutoff=burr_max_indeg,
    num_of_firms=num_firms,
    degree_sum=out_deg_seq_burr.sum(),
)
while (
    (in_deg_seq_burr.min().item() < 0)
    or (in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item() < 1.95)
    or (
        compute_hill_exponent(
            in_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
        < 2.3
    )
    or (
    compute_hill_exponent(
        in_deg_seq_burr,
        network_config.threshold_percentile_for_hill_deg,
        is_degree=True,
    ).item()
    > 2.4
    )
):
    in_deg_seq_burr = generate_burr_degrees_matching_sum(
        c_val=c_indeg,
        d_val=k_indeg,
        lam_val=lam_indeg,
        max_cutoff=burr_max_indeg,
        num_of_firms=num_firms,
        degree_sum=out_deg_seq_burr.sum(),
    )
    itr += 1
    print(itr)
in_deg_seq_burr.max(), in_deg_seq_burr.float().mean(), in_deg_seq_burr[
    in_deg_seq_burr > 0
].log().var(), compute_hill_exponent(
    in_deg_seq_burr, network_config.threshold_percentile_for_hill_deg, is_degree=True
)

Reorder the Burr out-degree sequence to induce correlation between the reordered Burr out-degrees and Burr in-degrees

In [ ]:
test_reordered_out_deg_seq_burr = triangular_wedge_matching(
    in_deg_seq=in_deg_seq_burr,
    out_deg_seq=out_deg_seq_burr,
    noise_strength=network_config.noise_amount_for_reordering_burrkiko,
)

Sample a binary network using the directed configuration model. Compute the corresponding in- and out-degree sequences from this binary network - we will use them to generate the weights later. 

In [ ]:
adj_mat_init = directed_configuration_model(
    in_deg_seq_burr, test_reordered_out_deg_seq_burr
)
in_deg_seq_init, out_deg_seq_init = compute_degrees_from_model(adj_mat_init)

Generate fitnesses

In [ ]:
in_str_seq_init, out_str_seq_init = generate_initial_strength_fitnesses(
    mean_in_str=network_config.mean_for_gen_lognorm_fi,
    mean_out_str=network_config.mean_for_gen_lognorm_fo,
    var_in_str=in_deg_seq_init[in_deg_seq_init > 0].log().var().item(),
    var_out_str=out_deg_seq_init[out_deg_seq_init > 0].log().var().item(),
    correlation=network_config.corr_lognorm_gen_fi_fo,
    num_of_firms=num_firms,
    monetary_unit=monetary_unit,
)

Load the IOT, and assign firms to industries

In [ ]:
data_iot_torch, data_iot_np, num_industries = load_input_output_table(
    data_path="data/",
    country_code=country_code,
    year=country_year,
    monetary_unit=monetary_unit,
)

# re-running assign_firms_to_industries will generate a different firm_to_industry_tensor. it is a random process.
firm_to_industry_tensor = assign_firms_to_industries(num_firms, data_iot_torch)

The next three cells are included so users can interact with the Optuna search, inspect its outputs, and get a feel for how the four theta parameters are estimated. For illustration, the search is run only once here. The theta values in the config folder were obtained by repeating the Optuna search two hundred times and are the ones used below to generate the weights.

In [ ]:
from network_generator.optuna_search.search_algo_fourparams import optuna_search_single_experiment

optuna_runner = optuna_search_single_experiment(
    in_deg=in_deg_seq_init,
    out_deg=out_deg_seq_init,
    in_str=in_str_seq_init,
    out_str=out_str_seq_init,
    adj_mat=adj_mat_init,
    data_iot=data_iot_torch,
    num_industries=num_industries,
    firm_to_industry=firm_to_industry_tensor,
    theta_k_in_upper_bound= training_config.theta_k_in_upper_bound,
    theta_k_in_lower_bound= training_config.theta_k_in_lower_bound,
    theta_k_out_upper_bound= training_config.theta_k_out_upper_bound,
    theta_k_out_lower_bound= training_config.theta_k_out_lower_bound,
    theta_f_in_upper_bound= training_config.theta_f_in_upper_bound,
    theta_f_in_lower_bound= training_config.theta_f_in_lower_bound,
    theta_f_out_upper_bound= training_config.theta_f_out_upper_bound,
    theta_f_out_lower_bound= training_config.theta_f_out_lower_bound,
    n_trials= training_config.n_trials,
    sampler_seed=None,
    target_config=target_config,
    network_config=network_config,
)

best_optuna_stats = optuna_runner.optimize()
optuna_trials_df = optuna_runner.get_trials_dataframe()
figures = optuna_runner.plot_figures(show = True)

In [ ]:
# Visualize the results of the Optuna search
best_optuna_stats

In [ ]:
# Present the tabular results of the Optuna search
optuna_trials_df

The pre-estimated theta parameters are used to compute the IOT-scaled weights, completing the final step of the generation process.

In [ ]:
wgt_mat_optim_pre_init = populate_weights_fourparams(
    in_deg_seq_init,
    out_deg_seq_init,
    in_str_seq_init,
    out_str_seq_init,
    adj_mat_init,
    network_config.theta_k_in_value,
    network_config.theta_k_out_value,
    network_config.theta_f_in_value,
    network_config.theta_f_out_value,
)
wgt_mat_optim_init = scale_by_iot(
    wgt_mat_optim_pre_init, data_iot_torch, firm_to_industry_tensor
)

In [ ]:
# Visualize the synthetic supply network
visualize_network(
    wgt_mat_optim_init,
    firm_to_industry_tensor,
    data_iot_torch,
    monetary_unit=monetary_unit,
    country_code=country_code,
    country_year=country_year,
)

In [ ]:
# Compute the network statistics
compute_network_stats(
    model=wgt_mat_optim_init,
    firm_to_industry=firm_to_industry_tensor,
    data_iot=data_iot_torch,
    num_of_industries=num_industries,
    network_config=network_config,
    target_config=target_config,
)

In [ ]:
# Save the generated network as .pth files, so that users can load them later without re-running the entire network generation process.
# In this case, the .pth files are saved in the paper/artifacts/main_paper folder.

# torch.save(wgt_mat_optim_init, "./paper/artifacts/main_paper/wgt_mat_optim_init.pth")
# torch.save(in_deg_seq_burr, "./paper/artifacts/main_paper/in_deg_seq_burr.pth")
# torch.save(out_deg_seq_burr, "./paper/artifacts/main_paper/out_deg_seq_burr.pth")
# torch.save(test_reordered_out_deg_seq_burr, "./paper/artifacts/main_paper/test_reordered_out_deg_seq_burr.pth")
# torch.save(adj_mat_init, "./paper/artifacts/main_paper/adj_mat_init.pth")
# torch.save(in_deg_seq_init, "./paper/artifacts/main_paper/in_deg_seq_init.pth")
# torch.save(out_deg_seq_init, "./paper/artifacts/main_paper/out_deg_seq_init.pth")
# torch.save(in_str_seq_init, "./paper/artifacts/main_paper/in_str_seq_init.pth")
# torch.save(out_str_seq_init, "./paper/artifacts/main_paper/out_str_seq_init.pth")
# torch.save(firm_to_industry_tensor, "./paper/artifacts/main_paper/firm_to_industry_tensor.pth")